In [1]:
# ==============================================================
# STEP 1 — CLEAN KAGGLE ENVIRONMENT SETUP
# ==============================================================

import os
import gc
import time
import json
import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp

print("=" * 70)
print("STEP 1 — CLEAN KAGGLE ENVIRONMENT")
print("=" * 70)

# --------------------------------------------------------------
# JAX CONFIGURATION
# --------------------------------------------------------------

jax.config.update("jax_enable_x64", True)

print("\nJAX version:")
print(jax.__version__)

print("\nJAX devices:")
for i, device in enumerate(jax.devices()):
    print(f"  [{i}] {device}")

print("\nX64 enabled:")
print(jax.config.read("jax_enable_x64"))

# --------------------------------------------------------------
# GPU CHECK
# --------------------------------------------------------------

gpu_devices = [
    d for d in jax.devices()
    if d.platform == "gpu"
]

print("\nGPU count visible to JAX:")
print(len(gpu_devices))

if len(gpu_devices) == 0:
    raise RuntimeError(
        "No GPU detected by JAX. Stop here and fix GPU setup."
    )

# We will use GPU 1 for the expensive attack stage
if len(gpu_devices) >= 2:
    ATTACK_DEVICE = gpu_devices[1]
else:
    ATTACK_DEVICE = gpu_devices[0]

print("\nSelected attack device:")
print(ATTACK_DEVICE)

# --------------------------------------------------------------
# WORKING DIRECTORY
# --------------------------------------------------------------

WORKDIR = "/kaggle/working"

os.makedirs(WORKDIR, exist_ok=True)

print("\nWorking directory:")
print(WORKDIR)

# --------------------------------------------------------------
# CHECK CURRENT FILES
# --------------------------------------------------------------

files = sorted(os.listdir(WORKDIR))

print("\nCurrent files in /kaggle/working:")

if len(files) == 0:
    print("  EMPTY")
else:
    for f in files:
        print(" ", f)

# --------------------------------------------------------------
# RESEARCH CONFIGURATION
# --------------------------------------------------------------

ID_DATASET = "CIFAR-100"
OOD_DATASET = "CIFAR-10"

IMAGE_SIZE = 384

N_TRAIN = 5000
N_TEST = 1000

N_ATTACK = 128

ATTACK_STEPS = 30
ATTACK_LR = 3e-4

TARGET_LINF = 1.0 / 255.0

print("\nResearch configuration:")
print("  ID dataset       :", ID_DATASET)
print("  OOD dataset      :", OOD_DATASET)
print("  Image size       :", IMAGE_SIZE)
print("  Train samples    :", N_TRAIN)
print("  Test samples     :", N_TEST)
print("  Attack samples   :", N_ATTACK)
print("  Attack steps     :", ATTACK_STEPS)
print("  Attack LR        :", ATTACK_LR)
print("  Target L-inf     :", TARGET_LINF)

# --------------------------------------------------------------
# BASIC NUMPY TEST
# --------------------------------------------------------------

test_array = np.zeros(
    (2, 2),
    dtype=np.float64
)

test_jax = jnp.asarray(
    test_array
)

print("\nBasic JAX test:")
print("  Shape  :", test_jax.shape)
print("  Dtype  :", test_jax.dtype)
print("  Device :", test_jax.device)

print("\n" + "=" * 70)
print("STEP 1 PASS")
print("=" * 70)

STEP 1 — CLEAN KAGGLE ENVIRONMENT

JAX version:
0.7.2

JAX devices:
  [0] cuda:0
  [1] cuda:1

X64 enabled:
True

GPU count visible to JAX:
2

Selected attack device:
cuda:1

Working directory:
/kaggle/working

Current files in /kaggle/working:
  __notebook__.ipynb

Research configuration:
  ID dataset       : CIFAR-100
  OOD dataset      : CIFAR-10
  Image size       : 384
  Train samples    : 5000
  Test samples     : 1000
  Attack samples   : 128
  Attack steps     : 30
  Attack LR        : 0.0003
  Target L-inf     : 0.00392156862745098

Basic JAX test:
  Shape  : (2, 2)
  Dtype  : float64
  Device : cuda:0

STEP 1 PASS


In [2]:
# ==============================================================
# STEP 2 — VIT + ORIGINAL REPOSITORY SETUP
# ==============================================================

import os
import sys
import subprocess
import shutil

print("=" * 70)
print("STEP 2 — VIT + REPOSITORY SETUP")
print("=" * 70)


# --------------------------------------------------------------
# PATHS
# --------------------------------------------------------------

WORKDIR = "/kaggle/working"

VIT_DIR = os.path.join(
    WORKDIR,
    "vision_transformer"
)

OOD_REPO_DIR = os.path.join(
    WORKDIR,
    "adversaries_to_OOD_detection"
)


# --------------------------------------------------------------
# HELPER
# --------------------------------------------------------------

def run_cmd(cmd):
    print("\n$", cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True
    )

    if result.stdout:
        print(result.stdout[-3000:])

    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(
            f"Command failed: {cmd}"
        )

    return result


# --------------------------------------------------------------
# 1. CLONE GOOGLE RESEARCH ViT
# --------------------------------------------------------------

if not os.path.exists(VIT_DIR):

    run_cmd(
        "git clone "
        "https://github.com/google-research/vision_transformer.git "
        f"{VIT_DIR}"
    )

    print("\nGoogle Research ViT cloned.")

else:

    print("\nGoogle Research ViT already exists.")


# --------------------------------------------------------------
# 2. CLONE PAPER REPOSITORY
# --------------------------------------------------------------

if not os.path.exists(OOD_REPO_DIR):

    run_cmd(
        "git clone "
        "https://github.com/stanislavfort/adversaries_to_OOD_detection.git "
        f"{OOD_REPO_DIR}"
    )

    print("\nPaper repository cloned.")

else:

    print("\nPaper repository already exists.")


# --------------------------------------------------------------
# 3. CHECK IMPORTANT FILES
# --------------------------------------------------------------

vit_models_path = os.path.join(
    VIT_DIR,
    "vit_jax",
    "models.py"
)

repo_models_path = os.path.join(
    OOD_REPO_DIR,
    "models.py"
)

print("\nImportant files:")

print(
    "Google ViT models.py:",
    os.path.exists(vit_models_path),
    vit_models_path
)

print(
    "Paper modified models.py:",
    os.path.exists(repo_models_path),
    repo_models_path
)


if not os.path.exists(vit_models_path):
    raise FileNotFoundError(
        "Google ViT models.py not found."
    )

if not os.path.exists(repo_models_path):
    raise FileNotFoundError(
        "Paper repository models.py not found."
    )


# --------------------------------------------------------------
# 4. BACKUP ORIGINAL GOOGLE ViT models.py
# --------------------------------------------------------------

backup_path = os.path.join(
    VIT_DIR,
    "vit_jax",
    "models_original.py"
)

if not os.path.exists(backup_path):

    shutil.copy2(
        vit_models_path,
        backup_path
    )

    print(
        "\nOriginal Google ViT models.py backed up."
    )

else:

    print(
        "\nOriginal models.py backup already exists."
    )


# --------------------------------------------------------------
# 5. COPY PAPER'S MODIFIED models.py
# --------------------------------------------------------------

shutil.copy2(
    repo_models_path,
    vit_models_path
)

print(
    "\nPaper's modified models.py copied into:"
)

print(vit_models_path)


# --------------------------------------------------------------
# 6. ADD ViT TO PYTHON PATH
# --------------------------------------------------------------

vit_parent = VIT_DIR

if vit_parent not in sys.path:
    sys.path.insert(
        0,
        vit_parent
    )

print("\nPython path updated.")


# --------------------------------------------------------------
# 7. CHECK REQUIRED PYTHON PACKAGES
# --------------------------------------------------------------

packages = [
    "jax",
    "flax",
    "ml_collections",
    "tensorflow",
    "PIL"
]

print("\nPackage availability:")

for package in packages:

    try:

        module = __import__(package)

        version = getattr(
            module,
            "__version__",
            "unknown"
        )

        print(
            f"  {package:<18} OK   {version}"
        )

    except Exception as e:

        print(
            f"  {package:<18} MISSING"
        )


# --------------------------------------------------------------
# 8. IMPORT ViT MODULE
# --------------------------------------------------------------

try:

    from vit_jax import models

    print(
        "\nViT models import: PASS"
    )

    print(
        "models.py:",
        models.__file__
    )

except Exception as e:

    print(
        "\nViT models import: FAILED"
    )

    print(
        repr(e)
    )

    raise


# --------------------------------------------------------------
# 9. CHECK MODIFIED MODEL CLASS
# --------------------------------------------------------------

required_classes = [
    "VisionTransformer",
    "VisionTransformer_prelogits"
]

print("\nRequired model classes:")

for name in required_classes:

    exists = hasattr(
        models,
        name
    )

    print(
        f"  {name:<30}",
        "PASS" if exists else "MISSING"
    )

    if not exists:
        raise AttributeError(
            f"{name} not found in modified models.py"
        )


# --------------------------------------------------------------
# FINAL
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 PASS")
print("=" * 70)

print("\nRepository:")
print(OOD_REPO_DIR)

print("\nViT:")
print(VIT_DIR)

print("\nModified models.py:")
print(vit_models_path)

STEP 2 — VIT + REPOSITORY SETUP

$ git clone https://github.com/google-research/vision_transformer.git /kaggle/working/vision_transformer

Google Research ViT cloned.

$ git clone https://github.com/stanislavfort/adversaries_to_OOD_detection.git /kaggle/working/adversaries_to_OOD_detection

Paper repository cloned.

Important files:
Google ViT models.py: True /kaggle/working/vision_transformer/vit_jax/models.py
Paper modified models.py: True /kaggle/working/adversaries_to_OOD_detection/models.py

Original Google ViT models.py backed up.

Paper's modified models.py copied into:
/kaggle/working/vision_transformer/vit_jax/models.py

Python path updated.

Package availability:
  jax                OK   0.7.2
  flax               OK   0.11.2
  ml_collections     OK   1.1.0
  tensorflow         OK   2.20.0
  PIL                OK   11.3.0

ViT models import: PASS
models.py: /kaggle/working/vision_transformer/vit_jax/models.py

Required model classes:
  VisionTransformer              PASS
  V

In [3]:
# ==============================================================
# STEP 3 — EXACT ViT-L/16 CIFAR-100 CHECKPOINT
# ==============================================================

import os
import urllib.request
import hashlib

print("=" * 70)
print("STEP 3 — EXACT ViT-L/16 CIFAR-100 CHECKPOINT")
print("=" * 70)

# --------------------------------------------------------------
# EXACT CHECKPOINT USED BY THE REPRODUCTION
# --------------------------------------------------------------

CHECKPOINT_NAME = (
    "L_16-i21k-300ep-lr_0.001-aug_strong1-"
    "wd_0.1-do_0.0-sd_0.0--"
    "cifar100-steps_2k-lr_0.01-res_384.npz"
)

CHECKPOINT_URL = (
    "https://storage.googleapis.com/"
    "vit_models/augreg/"
    + CHECKPOINT_NAME
)

CHECKPOINT_PATH = os.path.join(
    "/kaggle/working",
    CHECKPOINT_NAME
)

print("\nCheckpoint:")
print(CHECKPOINT_NAME)

print("\nTarget path:")
print(CHECKPOINT_PATH)

# --------------------------------------------------------------
# DOWNLOAD ONLY IF NOT ALREADY PRESENT
# --------------------------------------------------------------

if os.path.exists(CHECKPOINT_PATH):

    size_gb = (
        os.path.getsize(CHECKPOINT_PATH)
        / (1024 ** 3)
    )

    print("\nCheckpoint already exists.")
    print(
        f"Size: {size_gb:.3f} GB"
    )

else:

    print("\nCheckpoint not found.")
    print("Starting download...")
    print("This file is approximately 1.1 GB.")
    print()

    urllib.request.urlretrieve(
        CHECKPOINT_URL,
        CHECKPOINT_PATH
    )

    print("\nDownload complete.")

# --------------------------------------------------------------
# VERIFY FILE
# --------------------------------------------------------------

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        "Checkpoint download failed."
    )

file_size = os.path.getsize(
    CHECKPOINT_PATH
)

file_size_mb = (
    file_size / (1024 ** 2)
)

print("\nCheckpoint verification:")
print(
    "Exists :",
    os.path.exists(CHECKPOINT_PATH)
)

print(
    "Size   :",
    f"{file_size_mb:.2f} MB"
)

if file_size < 500 * 1024 * 1024:
    raise RuntimeError(
        "Checkpoint file is suspiciously small. "
        "Download may be incomplete."
    )

print("\n" + "=" * 70)
print("STEP 3 PASS")
print("=" * 70)

STEP 3 — EXACT ViT-L/16 CIFAR-100 CHECKPOINT

Checkpoint:
L_16-i21k-300ep-lr_0.001-aug_strong1-wd_0.1-do_0.0-sd_0.0--cifar100-steps_2k-lr_0.01-res_384.npz

Target path:
/kaggle/working/L_16-i21k-300ep-lr_0.001-aug_strong1-wd_0.1-do_0.0-sd_0.0--cifar100-steps_2k-lr_0.01-res_384.npz

Checkpoint not found.
Starting download...
This file is approximately 1.1 GB.


Download complete.

Checkpoint verification:
Exists : True
Size   : 1159.00 MB

STEP 3 PASS


In [4]:
# ==============================================================
# STEP 4A — INSPECT EXACT PAPER MODEL API
# ==============================================================

import inspect
from vit_jax import models

print("=" * 70)
print("STEP 4A — EXACT MODEL API CHECK")
print("=" * 70)

print("\nVisionTransformer signature:")
print(
    inspect.signature(
        models.VisionTransformer
    )
)

print("\nVisionTransformer_prelogits signature:")
print(
    inspect.signature(
        models.VisionTransformer_prelogits
    )
)

print("\n\nVisionTransformer source — first 120 lines:")
print(
    inspect.getsource(
        models.VisionTransformer
    )[:12000]
)

print("\n" + "=" * 70)
print("STEP 4A COMPLETE")
print("=" * 70)

STEP 4A — EXACT MODEL API CHECK

VisionTransformer signature:
(num_classes: int, patches: Any, transformer: Any, hidden_size: int, resnet: Optional[Any] = None, representation_size: Optional[int] = None, classifier: str = 'token', parent: Union[flax.linen.module.Module, flax.core.scope.Scope, flax.linen.module._Sentinel, NoneType] = <flax.linen.module._Sentinel object at 0x7e3cdf99df40>, name: Optional[str] = None) -> None

VisionTransformer_prelogits signature:
(num_classes: int, patches: Any, transformer: Any, hidden_size: int, resnet: Optional[Any] = None, representation_size: Optional[int] = None, classifier: str = 'token', parent: Union[flax.linen.module.Module, flax.core.scope.Scope, flax.linen.module._Sentinel, NoneType] = <flax.linen.module._Sentinel object at 0x7e3cdf99df40>, name: Optional[str] = None) -> None


VisionTransformer source — first 120 lines:
class VisionTransformer(nn.Module):
  """VisionTransformer."""

  num_classes: int
  patches: Any
  transformer: Any
  hid

In [5]:
# ==============================================================
# STEP 4B — EXACT ViT-L/16 MODEL INITIALIZATION
# ==============================================================

import os
import gc
import inspect
import numpy as np
import jax
import jax.numpy as jnp

from ml_collections import ConfigDict
from vit_jax import models
from vit_jax.configs import models as model_configs

print("=" * 70)
print("STEP 4B — EXACT ViT-L/16 MODEL INITIALIZATION")
print("=" * 70)


# --------------------------------------------------------------
# CHECKPOINT
# --------------------------------------------------------------

CHECKPOINT_NAME = (
    "L_16-i21k-300ep-lr_0.001-aug_strong1-"
    "wd_0.1-do_0.0-sd_0.0--"
    "cifar100-steps_2k-lr_0.01-res_384.npz"
)

CHECKPOINT_PATH = os.path.join(
    "/kaggle/working",
    CHECKPOINT_NAME
)

assert os.path.exists(
    CHECKPOINT_PATH
), "Checkpoint not found."


# --------------------------------------------------------------
# GET OFFICIAL ViT-L/16 CONFIG
# --------------------------------------------------------------

config = model_configs.get_l16_config()

print("\nOfficial L/16 configuration:")

print("Model name       :", config.model_name)
print("Patch size       :", config.patches.size)
print("Hidden size      :", config.hidden_size)
print("MLP dimension    :", config.transformer.mlp_dim)
print("Attention heads  :", config.transformer.num_heads)
print("Transformer layers:", config.transformer.num_layers)
print("Dropout          :", config.transformer.dropout_rate)
print("Attention dropout:",
      config.transformer.attention_dropout_rate)
print("Classifier       :", config.classifier)
print("Representation   :", config.representation_size)


# --------------------------------------------------------------
# REMOVE model_name
# --------------------------------------------------------------
#
# VisionTransformer does not accept model_name in this
# modified paper implementation.
#

if "model_name" in config:
    del config.model_name


# --------------------------------------------------------------
# CREATE CLASSIFIER MODEL
# --------------------------------------------------------------

print("\nCreating classifier model...")

model = models.VisionTransformer(
    num_classes=100,
    **config
)

print("Classifier model created.")


# --------------------------------------------------------------
# CREATE PRELOGITS MODEL
# --------------------------------------------------------------

print("\nCreating prelogits model...")

model_prelogits = models.VisionTransformer_prelogits(
    num_classes=100,
    **config
)

print("Prelogits model created.")


# --------------------------------------------------------------
# DUMMY INPUT
# --------------------------------------------------------------

dummy_input = jnp.zeros(
    (1, 384, 384, 3),
    dtype=jnp.float32
)

rng = jax.random.PRNGKey(0)


# --------------------------------------------------------------
# INITIALIZE CLASSIFIER
# --------------------------------------------------------------

print("\nInitializing classifier parameters...")

classifier_variables = model.init(
    rng,
    dummy_input,
    train=False
)

params = classifier_variables["params"]

print("Classifier parameters initialized.")


# --------------------------------------------------------------
# INITIALIZE PRELOGITS
# --------------------------------------------------------------

print("\nInitializing prelogits parameters...")

prelogits_variables = model_prelogits.init(
    rng,
    dummy_input,
    train=False
)

prelogits_params = prelogits_variables["params"]

print("Prelogits parameters initialized.")


# --------------------------------------------------------------
# PARAMETER TREE CHECK
# --------------------------------------------------------------

print("\nParameter tree check:")

print(
    "Classifier parameter keys:",
    list(params.keys())
)

print(
    "Prelogits parameter keys:",
    list(prelogits_params.keys())
)


# --------------------------------------------------------------
# FORWARD PASS BEFORE CHECKPOINT
# --------------------------------------------------------------

print("\nTesting classifier forward pass...")

test_logits = model.apply(
    {"params": params},
    dummy_input,
    train=False
)

print(
    "Classifier output:",
    test_logits.shape
)

print(
    "Classifier finite:",
    bool(
        np.all(
            np.isfinite(
                np.asarray(test_logits)
            )
        )
    )
)


print("\nTesting prelogits forward pass...")

test_features = model_prelogits.apply(
    {"params": prelogits_params},
    dummy_input,
    train=False
)

print(
    "Prelogits output:",
    test_features.shape
)

print(
    "Prelogits finite:",
    bool(
        np.all(
            np.isfinite(
                np.asarray(test_features)
            )
        )
    )
)


# --------------------------------------------------------------
# SHAPE ASSERTIONS
# --------------------------------------------------------------

assert test_logits.shape == (
    1,
    100
)

assert test_features.shape == (
    1,
    1024
)

assert np.all(
    np.isfinite(
        np.asarray(test_logits)
    )
)

assert np.all(
    np.isfinite(
        np.asarray(test_features)
    )
)


# --------------------------------------------------------------
# SAVE CONFIG FOR REPRODUCIBILITY
# --------------------------------------------------------------

config_dict = {
    "model": "ViT-L/16",
    "num_classes": 100,
    "input_resolution": 384,
    "patch_size": [16, 16],
    "hidden_size": 1024,
    "mlp_dim": 4096,
    "num_heads": 16,
    "num_layers": 24,
    "dropout_rate": 0.1,
    "attention_dropout_rate": 0.0,
    "classifier": "token",
    "representation_size": None,
}

import json

with open(
    "/kaggle/working/vit_l16_config.json",
    "w"
) as f:
    json.dump(
        config_dict,
        f,
        indent=2
    )


# --------------------------------------------------------------
# CLEAN TEMP VARIABLES
# --------------------------------------------------------------

del dummy_input
del classifier_variables
del prelogits_variables

gc.collect()


# --------------------------------------------------------------
# FINAL
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4B PASS")
print("=" * 70)

print("\nViT-L/16:")
print("  Input       : 384 × 384 × 3")
print("  Classes     : 100")
print("  Hidden      : 1024")
print("  Layers      : 24")
print("  Heads       : 16")
print("  MLP         : 4096")
print("  Patch       : 16 × 16")

print("\nOutputs:")
print("  Classifier  :", test_logits.shape)
print("  Prelogits   :", test_features.shape)

print("\nConfig saved:")
print("/kaggle/working/vit_l16_config.json")

STEP 4B — EXACT ViT-L/16 MODEL INITIALIZATION

Official L/16 configuration:
Model name       : ViT-L_16
Patch size       : (16, 16)
Hidden size      : 1024
MLP dimension    : 4096
Attention heads  : 16
Transformer layers: 24
Dropout          : 0.1
Attention dropout: 0.0
Classifier       : token
Representation   : None

Creating classifier model...
Classifier model created.

Creating prelogits model...
Prelogits model created.

Initializing classifier parameters...
Classifier parameters initialized.

Initializing prelogits parameters...
Prelogits parameters initialized.

Parameter tree check:
Classifier parameter keys: ['embedding', 'cls', 'Transformer', 'head']
Prelogits parameter keys: ['embedding', 'cls', 'Transformer', 'head']

Testing classifier forward pass...
Classifier output: (1, 100)
Classifier finite: True

Testing prelogits forward pass...
Prelogits output: (1, 1024)
Prelogits finite: True

STEP 4B PASS

ViT-L/16:
  Input       : 384 × 384 × 3
  Classes     : 100
  Hidden   